# Business Understanding: Analisis dan Pemantauan Kualitas Udara Lokal

## 1. Latar Belakang
Seiring dengan peningkatan urbanisasi, aktivitas industri, dan volume kendaraan bermotor, kualitas udara di berbagai daerah mengalami penurunan yang signifikan. Penurunan kualitas udara ini berbanding lurus dengan peningkatan risiko kesehatan masyarakat dan kerusakan lingkungan. Oleh karena itu, tugas untuk mengetahui, menganalisis, dan memantau kualitas udara di daerah masing-masing menjadi sangat krusial sebagai langkah mitigasi awal dan dasar pengambilan keputusan strategis.

## 2. Tujuan Bisnis (Business Objectives)
Tugas pemantauan dan analisis kualitas udara (khususnya untuk polutan seperti NO2, CO, dan SO2) memiliki beberapa tujuan utama:
* **Pemantauan Kondisi Lingkungan:** Mendapatkan gambaran yang akurat, terukur, dan *real-time* atau historis mengenai tingkat polusi udara di suatu wilayah spesifik (misalnya tingkat emisi di tingkat kabupaten/kota).
* **Identifikasi Tren dan Pola:** Mengetahui kapan dan di mana lonjakan polutan sering terjadi (misalnya saat jam sibuk, musim kemarau, atau di sekitar kawasan industri).
* **Peringatan Dini (Early Warning System):** Menciptakan dasar sistem informasi yang dapat memberikan peringatan kepada masyarakat ketika kualitas udara mencapai level berbahaya.
* **Evaluasi Kebijakan:** Mengukur efektivitas kebijakan lingkungan yang sudah berjalan, seperti pembatasan kendaraan bermotor atau penegakan standar emisi pabrik.

## 3. Manfaat (Business Benefits)
Mengetahui kualitas udara di daerah masing-masing memberikan dampak positif (manfaat) bagi berbagai pemangku kepentingan (*stakeholders*):

### A. Bagi Masyarakat Umum (Public & Health)
* **Perlindungan Kesehatan:** Masyarakat dapat menyesuaikan aktivitas harian mereka (misalnya mengurangi aktivitas di luar ruangan atau menggunakan masker) saat tingkat polusi tinggi, sehingga mengurangi risiko penyakit pernapasan (ISPA, asma).
* **Peningkatan Kualitas Hidup:** Udara yang terpantau dan dikelola dengan baik akan menciptakan lingkungan tempat tinggal yang lebih sehat dan nyaman.

### B. Bagi Pemerintah dan Pembuat Kebijakan (Government & Policy Makers)
* **Dasar Pembuatan Regulasi (Data-Driven Policy):** Memberikan bukti empiris (*evidence-based*) untuk merumuskan kebijakan publik, seperti penataan kawasan industri, perluasan ruang terbuka hijau (RTH), atau manajemen rekayasa lalu lintas.
* **Target Pembangunan Berkelanjutan (SDGs):** Membantu pemerintah daerah dalam mencapai target *Sustainable Development Goals*, khususnya terkait Kota dan Komunitas yang Berkelanjutan (SDG 11) serta Penanganan Perubahan Iklim (SDG 13).

### C. Bagi Sektor Bisnis dan Industri (Private Sector)
* **Kepatuhan Lingkungan (ESG Compliance):** Membantu perusahaan dalam memantau emisi mereka sendiri untuk memastikan kepatuhan terhadap standar regulasi lingkungan yang berlaku (Environmental, Social, and Governance).
* **Peluang Inovasi:** Membuka peluang pasar baru untuk produk kesehatan dan lingkungan, seperti alat pembersih udara (*air purifier*), masker kesehatan khusus, atau sistem ventilasi gedung yang cerdas.

### D. Bagi Akademisi dan Peneliti
* **Ketersediaan Data Riset:** Menyediakan dataset yang valid dan kaya untuk keperluan penelitian lanjutan terkait dampak perubahan iklim, kesehatan masyarakat, dan meteorologi.

Menginstal pustaka Python openeo beserta seluruh dependensinya (seperti pystac, xarray, dll.) menggunakan perintah pip.

In [1]:
pip install openeo

^C


Note: you may need to restart the kernel to use updated packages.


Mengimpor pustaka openeo ke dalam environment agar fungsinya dapat digunakan.

In [2]:
import openeo

Membuat koneksi ke server OpenEO Copernicus Data Space (openeo.dataspace.copernicus.eu) dan melakukan proses autentikasi pengguna menggunakan metode OpenID Connect (OIDC).

In [3]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=FIWF-MKAN 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


Mendefinisikan Area of Interest (AOI) berupa poligon koordinat, lalu memuat koleksi data satelit SENTINEL_5P_L2 untuk variabel gas NO2, CO, dan SO2 pada rentang waktu 1 Januari 2025 hingga 26 Agustus 2026. Data tersebut kemudian diagregasi menjadi rata-rata harian secara temporal, serta dirata-rata secara spasial sesuai batas area AOI.

In [11]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [113.09, -6.89],
            [112.68, -6.89],
            [112.68, -7.20],
            [113.09, -7.20],
            [113.09, -6.89],
        ]
    ]
}

s5NO2 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-01-01", "2026-08-26"],
    spatial_extent={
        "west": 112.68,
        "south": -7.20,
        "east": 113.09,
        "north": -6.89
    },
    bands=["NO2"],
)

s5CO = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-01-01", "2026-08-26"],
    spatial_extent={
        "west": 112.68,
        "south": -7.20,
        "east": 113.09,
        "north": -6.89
    },
    bands=["CO"],
)

s5SO2 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-01-01", "2026-08-26"],
    spatial_extent={
        "west": 112.68,
        "south": -7.20,
        "east": 113.09,
        "north": -6.89
    },
    bands=["SO2"],
)

# Now aggregate by day to avoid having multiple data per day
s5p_NO2_daily = s5NO2.aggregate_temporal_period(reducer="mean", period="day")
s5p_CO_daily = s5CO.aggregate_temporal_period(reducer="mean", period="day")
s5p_SO2_daily = s5SO2.aggregate_temporal_period(reducer="mean", period="day")

# Now create a spatial aggregation to generate mean timeseries data
s5p_NO2_aoi = s5p_NO2_daily.aggregate_spatial(reducer="mean", geometries=aoi)
s5p_CO_aoi = s5p_CO_daily.aggregate_spatial(reducer="mean", geometries=aoi)
s5p_SO2_aoi = s5p_SO2_daily.aggregate_spatial(reducer="mean", geometries=aoi)


Menjalankan proses (batch job) di server untuk mengeksekusi data NO2, CO, SO2 yang telah diagregasi, dan mengunduh hasilnya ke dalam file format NetCDF

In [13]:
job = s5p_NO2_aoi.execute_batch(title="NO2 in Bangkalan terkini", outputfile="NO2Bangkalan Terkini.nc")

0:00:00 Job 'j-260827085655455789baa71b8342d0c9': send 'start'
0:00:03 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:00:08 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:00:14 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:00:22 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:00:32 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:00:45 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:01:00 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:01:20 Job 'j-260827085655455789baa71b8342d0c9': queued (progress 0%)
0:01:44 Job 'j-260827085655455789baa71b8342d0c9': running (progress N/A)
0:02:14 Job 'j-260827085655455789baa71b8342d0c9': running (progress N/A)
0:02:51 Job 'j-260827085655455789baa71b8342d0c9': running (progress N/A)
0:03:38 Job 'j-260827085655455789baa71b8342d0c9': running (progress N/A)
0:04:36 Job 'j-260827085655455789baa71b8342d0c9': running (progress N/A)
0:05

In [14]:
job = s5CO.execute_batch(title="CO in Bangkalan terkini", outputfile="COBangkalan Terkini.nc")

0:00:00 Job 'j-260827090238499aade5f9d4a5240270': send 'start'
0:00:03 Job 'j-260827090238499aade5f9d4a5240270': created (progress 0%)
0:00:09 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:00:15 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:00:23 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:00:33 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:00:45 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:01:01 Job 'j-260827090238499aade5f9d4a5240270': queued (progress 0%)
0:01:20 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0:01:44 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0:02:14 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0:02:52 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0:03:38 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0:04:37 Job 'j-260827090238499aade5f9d4a5240270': running (progress N/A)
0

In [15]:
job = s5SO2.execute_batch(title="SO2 in Bangkalan terkini", outputfile="SO2Bangkalan Terkini.nc")

0:00:00 Job 'j-2608270908214bc5973a8b286d083ad8': send 'start'
0:00:02 Job 'j-2608270908214bc5973a8b286d083ad8': created (progress 0%)
0:00:07 Job 'j-2608270908214bc5973a8b286d083ad8': queued (progress 0%)
0:00:14 Job 'j-2608270908214bc5973a8b286d083ad8': queued (progress 0%)
0:00:22 Job 'j-2608270908214bc5973a8b286d083ad8': queued (progress 0%)
0:00:32 Job 'j-2608270908214bc5973a8b286d083ad8': queued (progress 0%)
0:00:44 Job 'j-2608270908214bc5973a8b286d083ad8': queued (progress 0%)
0:01:00 Job 'j-2608270908214bc5973a8b286d083ad8': running (progress N/A)
0:01:19 Job 'j-2608270908214bc5973a8b286d083ad8': running (progress N/A)
0:01:43 Job 'j-2608270908214bc5973a8b286d083ad8': running (progress N/A)
0:02:13 Job 'j-2608270908214bc5973a8b286d083ad8': running (progress N/A)
0:02:51 Job 'j-2608270908214bc5973a8b286d083ad8': running (progress N/A)
0:03:38 Job 'j-2608270908214bc5973a8b286d083ad8': finished (progress 100%)


Menginstal pustaka netCDF4 yang diperlukan untuk membaca dan memanipulasi file dengan ekstensi .nc di Python.

In [ ]:
pip install netCDF4

Mengimpor pustaka analisis data (netCDF4, numpy, pandas) dan membaca ketiga file NetCDF (NO2, CO, SO2) yang baru saja diunduh ke dalam variabel dataset masing-masing (dsNO2, dsCO, dsSO2)

In [34]:
import netCDF4
import numpy as np
import pandas as pd

dsNO2 = netCDF4.Dataset("NO2Bangkalan Terkini.nc")
dsCO = netCDF4.Dataset("COBangkalan Terkini.nc")
dsSO2 = netCDF4.Dataset("SO2Bangkalan Terkini.nc")

Mencetak (menampilkan) daftar keys atau variabel penyusun apa saja yang terdapat di dalam struktur data dataset CO.

In [35]:
print("📦 Variabel dalam file:")
print(dsCO.variables.keys())

📦 Variabel dalam file:
dict_keys(['t', 'x', 'y', 'crs', 'CO'])


Mengekstrak nilai kadar NO2, CO, SO2 dan nilai waktu (t) dari dataset NO2. Kode ini kemudian mengubah format waktu menjadi tanggal string (YYYY-MM-DD), menyusunnya menjadi struktur tabel DataFrame menggunakan Pandas, dan mengekspornya menjadi file CSV.

In [50]:
no2 = dsNO2.variables["NO2"][:].squeeze()
timeNO2 = dsNO2.variables["t"][:]

try:
    time_units = dsNO2.variables["t"].units
    dates = netCDF4.num2date(timeNO2, units=time_units)
except Exception:
    dates = timeNO2  # fallback kalau tidak ada units

new_dates = []
for i in range(len(dates)):
    # ubah format datetime
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)

df = pd.DataFrame({
    "date": new_dates,
    "NO2": no2
})

# Simpan ke CSV
df.to_csv("NO2_Bangkalan_timeseries_terkini.csv", index=False)

In [47]:
co = dsCO.variables["CO"][:].mean(axis=(1, 2))
timeCO = dsCO.variables["t"][:]

try:
    time_units = dsCO.variables["t"].units
    dates = netCDF4.num2date(timeCO, units=time_units)
except Exception:
    dates = timeCO  # fallback kalau tidak ada units

new_dates = []
for i in range(len(dates)):
    # ubah format datetime
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)

df = pd.DataFrame({
    "date": new_dates,
    "CO": co
})

# Simpan ke CSV
df.to_csv("CO_Bangkalan_timeseries_terkini.csv", index=False)

In [48]:
so2 = dsSO2.variables["SO2"][:].mean(axis=(1, 2))
timeSO2 = dsSO2.variables["t"][:]

try:
    time_units = dsSO2.variables["t"].units
    dates = netCDF4.num2date(timeSO2, units=time_units)
except Exception:
    dates = timeSO2  # fallback kalau tidak ada units

new_dates = []
for i in range(len(dates)):
    # ubah format datetime
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)

df = pd.DataFrame({
    "date": new_dates,
    "SO2": so2
})

# Simpan ke CSV
df.to_csv("SO2_Bangkalan_timeseries_terkini.csv", index=False)